# Notebook B: Final Training + Dual Ensemble Comparison

This notebook is the fixed-configuration evaluation entry point for the thesis and for Richard.

It does four things:
1. Loads frozen best configs from Notebook A, but overrides `GRU-D` with the thesis strong-version hyperparameters.
2. Runs fixed 5-fold CV for `LSTM`, `GRU-D`, `SVM`, and `LR`.
3. Tunes deterministic ensemble weights for both `GRU-D + LSTM + SVM` and `GRU-D + LSTM + LR` using a two-stage search:
   first a `0.05` global grid, then a `0.01` local refinement around the coarse optimum.
4. Saves the final comparison tables and the selected ensemble weights.

It does **not** run any random search.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate the project root containing pyproject.toml.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dual_ensemble_workflow import (
    configure_tensorflow_runtime,
    DEFAULT_MIN_SPECIFICITY,
    DEFAULT_WEIGHT_STEP,
    comparison_table,
    load_workflow_context,
    run_grud_cv,
    run_lr_cv,
    run_lstm_cv,
    run_svm_cv,
    save_json,
    search_ensemble_weights_with_local_refinement,
    evaluate_ensemble,
    to_python,
)
from paths import RESULTS_DIR

In [2]:
SEARCH_OUTPUT_DIR = RESULTS_DIR / "dual_ensemble_notebooks" / "hyperparameter_search"
FINAL_OUTPUT_DIR = RESULTS_DIR / "dual_ensemble_notebooks" / "final_training"
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
USE_TF_GPU_FOR_FINAL_LSTM = False  # set to True only if your TensorFlow Metal runs stably

CONFIG_PATH = SEARCH_OUTPUT_DIR / "best_model_configs.json"
if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Missing frozen config file: {CONFIG_PATH}. Run Notebook A first."
    )

THESIS_GRUD_CONFIG = {
    "hidden_size": 64,
    "dropout": 0.3,
    "lr": 0.001,
    "batch_size": 64,
    "weight_decay": 0.0001,
    "max_epochs": 100,
    "patience": 15,
}

COARSE_WEIGHT_STEP = DEFAULT_WEIGHT_STEP
REFINE_WEIGHT_STEP = 0.01
REFINE_RADIUS = DEFAULT_WEIGHT_STEP
MIN_SPECIFICITY = DEFAULT_MIN_SPECIFICITY

In [3]:
tf_runtime = configure_tensorflow_runtime(use_gpu=USE_TF_GPU_FOR_FINAL_LSTM)
with CONFIG_PATH.open() as handle:
    config_payload = json.load(handle)

best_configs = config_payload["best_configs"]
best_configs["GRU-D"] = THESIS_GRUD_CONFIG
context = load_workflow_context()

display(pd.DataFrame(
    [
        {"Model": model_name, "Best Config": json.dumps(cfg)}
        for model_name, cfg in best_configs.items()
    ]
))
print(f"TensorFlow runtime: {tf_runtime}")
print("Notebook B is using frozen configs only. No random search is executed here.")
print("GRU-D is explicitly overridden with the thesis strong-version hyperparameters.")

Loading patient files...
  Loaded 4000 patients, 37 ts variables


,Model,Best Config
0,LSTM,"{""units"": [64], ""dropout"": 0.3, ""lr"": 0.0005, ""batch_size"": 32, ""max_epochs"": 40, ""patience"": 8}"
1,GRU-D,"{""hidden_size"": 64, ""dropout"": 0.3, ""lr"": 0.001, ""batch_size"": 64, ""weight_decay"": 0.0001, ""max_epochs"": 100, ""patie..."
2,SVM,"{""kernel"": ""linear"", ""C"": 0.01}"
3,LR,"{""C"": 0.01, ""penalty"": ""l2"", ""solver"": ""liblinear"", ""max_iter"": 5000}"


TensorFlow runtime: {'requested_gpu': False, 'physical_gpu_count': 0, 'logical_gpu_count': 0, 'active_device': 'CPU'}
Notebook B is using frozen configs only. No random search is executed here.
GRU-D is explicitly overridden with the thesis strong-version hyperparameters.


In [4]:
lstm_results = run_lstm_cv(best_configs["LSTM"], context=context, keras_verbose=0)
grud_results = run_grud_cv(best_configs["GRU-D"], context=context, log_every=5)
svm_results = run_svm_cv(best_configs["SVM"], context=context)
lr_results = run_lr_cv(best_configs["LR"], context=context)

single_model_results = {
    "LSTM": lstm_results,
    "GRU-D": grud_results,
    "SVM": svm_results,
    "LR": lr_results,
}


LSTM 5-Fold CV
Fold 1/5

  AUROC=0.8266 | AUPRC=0.4484 | F1=0.4400
Fold 2/5
  AUROC=0.8096 | AUPRC=0.4520 | F1=0.4364
Fold 3/5
  AUROC=0.8540 | AUPRC=0.5017 | F1=0.4128
Fold 4/5
  AUROC=0.8303 | AUPRC=0.4855 | F1=0.4034
Fold 5/5
  AUROC=0.8312 | AUPRC=0.4461 | F1=0.4330

5-Fold CV Summary (mean ± std)
  AUROC         : 0.8303 ± 0.0142
  AUPRC         : 0.4667 ± 0.0226
  F1            : 0.4251 ± 0.0144
  Sensitivity   : 0.8213 ± 0.0955
  Specificity   : 0.6695 ± 0.0711
  PPV           : 0.2899 ± 0.0244
  NPV           : 0.9607 ± 0.0160
  Brier         : 0.0962 ± 0.0020
  Threshold     : 0.0920 ± 0.0334

GRU-D 5-Fold CV
Fold 1/5
    epoch 001/100 | val AUPRC=0.4347 | best=-1.0000
    epoch 005/100 | val AUPRC=0.4710 | best=0.4923
    epoch 010/100 | val AUPRC=0.4184 | best=0.4923
    epoch 015/100 | val AUPRC=0.4103 | best=0.4923
    early stop at epoch 17 (best epoch 2)
  AUROC=0.8331 | AUPRC=0.4890 | F1=0.4592
Fold 2/5
    epoch 001/100 | val AUPRC=0.4838 | best=-1.0000
    epoch 005/

In [5]:
single_model_table = comparison_table(single_model_results, {})
display(single_model_table)

,Model,Metric,Mean,Std
0,LSTM,AUROC,0.830342,0.014157
1,LSTM,AUPRC,0.466739,0.022579
2,LSTM,F1,0.425124,0.014377
3,LSTM,Sensitivity,0.821261,0.095488
4,LSTM,Specificity,0.669461,0.071144
5,GRU-D,AUROC,0.830279,0.011079
6,GRU-D,AUPRC,0.475060,0.029489
7,GRU-D,F1,0.443541,0.018529
8,GRU-D,Sensitivity,0.730975,0.079716
9,GRU-D,Specificity,0.747231,0.051319


In [7]:
import pickle
from paths import RESULTS_DIR

SAVE_DIR = RESULTS_DIR / "dual_ensemble_notebooks" / "final_training"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ---- 4 个基模型的 CV 结果（含每折 val/test 概率、y、阈值） ----
with open(SAVE_DIR / "lstm_results.pkl", "wb") as f: pickle.dump(lstm_results, f)
with open(SAVE_DIR / "grud_results.pkl", "wb") as f: pickle.dump(grud_results, f)
with open(SAVE_DIR / "svm_results.pkl",  "wb") as f: pickle.dump(svm_results,  f)
with open(SAVE_DIR / "lr_results.pkl",   "wb") as f: pickle.dump(lr_results,   f)


In [11]:
FIXED_W_SVM = {"GRU-D": 0.06, "LSTM": 0.36, "SVM": 0.58}
FIXED_W_LR  = {"GRU-D": 0.05, "LSTM": 0.50, "LR":  0.45}

component_svm = {"GRU-D": grud_results, "LSTM": lstm_results, "SVM": svm_results}
component_lr  = {"GRU-D": grud_results, "LSTM": lstm_results, "LR":  lr_results}

ensemble_svm_results = evaluate_ensemble(
    component_results=component_svm,
    model_order=("GRU-D", "LSTM", "SVM"),
    weights=FIXED_W_SVM,
    label="Ensemble GRU-D + LSTM + SVM (fixed weights)",
)
ensemble_lr_results = evaluate_ensemble(
    component_results=component_lr,
    model_order=("GRU-D", "LSTM", "LR"),
    weights=FIXED_W_LR,
    label="Ensemble GRU-D + LSTM + LR (fixed weights)",
)

with open(SAVE_DIR / "ensemble_svm_results.pkl", "wb") as f: pickle.dump(ensemble_svm_results, f)
with open(SAVE_DIR / "ensemble_lr_results.pkl",  "wb") as f: pickle.dump(ensemble_lr_results,  f)




Ensemble GRU-D + LSTM + SVM (fixed weights)
Fold 1/5 | AUROC=0.8405 | AUPRC=0.4813 | F1=0.4279
Fold 2/5 | AUROC=0.8540 | AUPRC=0.5007 | F1=0.4912
Fold 3/5 | AUROC=0.8754 | AUPRC=0.5501 | F1=0.4234
Fold 4/5 | AUROC=0.8474 | AUPRC=0.5225 | F1=0.4271
Fold 5/5 | AUROC=0.8547 | AUPRC=0.5018 | F1=0.4227

5-Fold CV Summary (mean ± std)
  AUROC         : 0.8544 ± 0.0117
  AUPRC         : 0.5113 ± 0.0234
  F1            : 0.4384 ± 0.0265
  Sensitivity   : 0.8646 ± 0.0651
  Specificity   : 0.6613 ± 0.0673
  PPV           : 0.2963 ± 0.0340
  NPV           : 0.9694 ± 0.0114
  Brier         : 0.0917 ± 0.0010
  Threshold     : 0.1153 ± 0.0189

Ensemble GRU-D + LSTM + LR (fixed weights)
Fold 1/5 | AUROC=0.8432 | AUPRC=0.4820 | F1=0.4286
Fold 2/5 | AUROC=0.8534 | AUPRC=0.4967 | F1=0.4633
Fold 3/5 | AUROC=0.8736 | AUPRC=0.5371 | F1=0.4615
Fold 4/5 | AUROC=0.8492 | AUPRC=0.5305 | F1=0.4476
Fold 5/5 | AUROC=0.8584 | AUPRC=0.5016 | F1=0.4135

5-Fold CV Summary (mean ± std)
  AUROC         : 0.8556 ± 0.01

In [13]:
#inspection
import pickle
import numpy as np
from paths import RESULTS_DIR
from sklearn.metrics import roc_auc_score

SAVE_DIR = RESULTS_DIR / "dual_ensemble_notebooks" / "final_training"

def load(name):
    with open(SAVE_DIR / f"{name}.pkl", "rb") as f:
        return pickle.load(f)

lstm_results        = load("lstm_results")
grud_results        = load("grud_results")
svm_results         = load("svm_results")
lr_results          = load("lr_results")
ensemble_svm_results = load("ensemble_svm_results")
ensemble_lr_results  = load("ensemble_lr_results")

# 逐折打印 AUROC，确认和训练时输出一致
for name, res in [
    ("LSTM", lstm_results), ("GRU-D", grud_results),
    ("SVM",  svm_results),  ("LR",    lr_results),
    ("Ensemble-SVM", ensemble_svm_results),
    ("Ensemble-LR",  ensemble_lr_results),
]:
    aurocs = []
    for fo in res["fold_outputs"]:
        y = fo["y_test"]
        # 基模型 key 是 test_prob_cal，ensemble 是 test_prob
        p = fo.get("test_prob_cal", fo.get("test_prob"))
        aurocs.append(roc_auc_score(y, p))
    print(f"{name:15s}  per-fold AUROC = {[f'{a:.4f}' for a in aurocs]}  mean={np.mean(aurocs):.4f}")


LSTM             per-fold AUROC = ['0.8266', '0.8096', '0.8540', '0.8303', '0.8312']  mean=0.8303
GRU-D            per-fold AUROC = ['0.8331', '0.8383', '0.8118', '0.8433', '0.8248']  mean=0.8303
SVM              per-fold AUROC = ['0.8112', '0.8437', '0.8600', '0.8139', '0.8426']  mean=0.8343
LR               per-fold AUROC = ['0.8227', '0.8466', '0.8641', '0.8268', '0.8544']  mean=0.8429
Ensemble-SVM     per-fold AUROC = ['0.8405', '0.8540', '0.8754', '0.8474', '0.8547']  mean=0.8544
Ensemble-LR      per-fold AUROC = ['0.8432', '0.8534', '0.8736', '0.8492', '0.8584']  mean=0.8556


In [8]:
svm_components = {
    "GRU-D": grud_results,
    "LSTM": lstm_results,
    "SVM": svm_results,
}
svm_weight_result, svm_coarse_weight_grid, svm_refined_weight_grid = search_ensemble_weights_with_local_refinement(
    svm_components,
    model_order=("GRU-D", "LSTM", "SVM"),
    coarse_step=COARSE_WEIGHT_STEP,
    refine_step=REFINE_WEIGHT_STEP,
    refine_radius=REFINE_RADIUS,
    min_specificity=MIN_SPECIFICITY,
)
ensemble_svm_results = evaluate_ensemble(
    svm_components,
    model_order=("GRU-D", "LSTM", "SVM"),
    weights=svm_weight_result["weights"],
    label="Ensemble (GRU-D + LSTM + SVM)",
)

display(svm_coarse_weight_grid.head(10))
display(svm_refined_weight_grid.head(10))
display(pd.DataFrame([svm_weight_result["weights"]]))

with open(SAVE_DIR / "ensemble_svm_results.pkl", "wb") as f: pickle.dump(ensemble_svm_results, f)


Weight Search: GRU-D + LSTM + SVM
Best weights: {'GRU-D': 0.05, 'LSTM': 0.35000000000000003, 'SVM': 0.6000000000000001} | val Sensitivity=0.8732 | val Specificity=0.6786 | val AUPRC=0.4627
Best weights: {'GRU-D': 0.06, 'LSTM': 0.36, 'SVM': 0.58} | val Sensitivity=0.8822 | val Specificity=0.6694 | val AUPRC=0.4636
Coarse best: {'GRU-D': 0.05, 'LSTM': 0.35000000000000003, 'SVM': 0.6000000000000001} | val Sensitivity=0.8732 | val Specificity=0.6786
Refined best: {'GRU-D': 0.06, 'LSTM': 0.36, 'SVM': 0.58} | val Sensitivity=0.8822 | val Specificity=0.6694 | val AUPRC=0.4636

Ensemble (GRU-D + LSTM + SVM)
Fold 1/5 | AUROC=0.8405 | AUPRC=0.4813 | F1=0.4279
Fold 2/5 | AUROC=0.8540 | AUPRC=0.5007 | F1=0.4912
Fold 3/5 | AUROC=0.8754 | AUPRC=0.5501 | F1=0.4234
Fold 4/5 | AUROC=0.8474 | AUPRC=0.5225 | F1=0.4271
Fold 5/5 | AUROC=0.8547 | AUPRC=0.5018 | F1=0.4227

5-Fold CV Summary (mean ± std)
  AUROC         : 0.8544 ± 0.0117
  AUPRC         : 0.5113 ± 0.0234
  F1            : 0.4384 ± 0.0265
  S

,w_1,w_2,w_3,mean_sensitivity,mean_specificity,mean_auprc,mean_auroc,mean_f1
0,0.05,0.35,0.60,0.873225,0.678611,0.462749,0.834836,0.451639
1,0.10,0.30,0.60,0.873180,0.681986,0.465210,0.835704,0.453774
2,0.05,0.80,0.15,0.861239,0.668474,0.470397,0.830842,0.443298
3,0.00,0.75,0.25,0.861239,0.676686,0.466382,0.831657,0.448845
4,0.00,0.85,0.15,0.861239,0.663154,0.464428,0.828526,0.439341
5,0.00,0.80,0.20,0.861239,0.670889,0.463862,0.830330,0.445256
6,0.00,0.90,0.10,0.861239,0.659769,0.459875,0.826335,0.435661
7,0.15,0.40,0.45,0.861104,0.688270,0.473853,0.837572,0.454375
8,0.25,0.35,0.40,0.861058,0.693582,0.482318,0.838932,0.459787
9,0.20,0.45,0.35,0.858209,0.692155,0.480710,0.838306,0.455929


,w_1,w_2,w_3,mean_sensitivity,mean_specificity,mean_auprc,mean_auroc,mean_f1
0,0.06,0.36,0.58,0.882225,0.669417,0.463570,0.835493,0.448344
1,0.05,0.36,0.59,0.882225,0.667967,0.463000,0.835150,0.447473
2,0.06,0.35,0.59,0.879195,0.671832,0.463141,0.835362,0.448945
3,0.06,0.34,0.60,0.879195,0.672799,0.463067,0.835135,0.449633
4,0.04,0.37,0.59,0.876255,0.674746,0.462689,0.835025,0.450144
5,0.06,0.31,0.63,0.876255,0.675232,0.462240,0.834740,0.450902
6,0.04,0.36,0.60,0.876255,0.673297,0.461850,0.834689,0.449244
7,0.03,0.37,0.60,0.876255,0.673297,0.460979,0.834463,0.449336
8,0.03,0.36,0.61,0.876255,0.674265,0.460782,0.834287,0.450196
9,0.09,0.32,0.59,0.876165,0.677629,0.464652,0.835676,0.451706


,GRU-D,LSTM,SVM
0,0.06,0.36,0.58


In [9]:
lr_components = {
    "GRU-D": grud_results,
    "LSTM": lstm_results,
    "LR": lr_results,
}
lr_weight_result, lr_coarse_weight_grid, lr_refined_weight_grid = search_ensemble_weights_with_local_refinement(
    lr_components,
    model_order=("GRU-D", "LSTM", "LR"),
    coarse_step=COARSE_WEIGHT_STEP,
    refine_step=REFINE_WEIGHT_STEP,
    refine_radius=REFINE_RADIUS,
    min_specificity=MIN_SPECIFICITY,
)
ensemble_lr_results = evaluate_ensemble(
    lr_components,
    model_order=("GRU-D", "LSTM", "LR"),
    weights=lr_weight_result["weights"],
    label="Ensemble (GRU-D + LSTM + LR)",
)

display(lr_coarse_weight_grid.head(10))
display(lr_refined_weight_grid.head(10))
display(pd.DataFrame([lr_weight_result["weights"]]))

with open(SAVE_DIR / "ensemble_lr_results.pkl",  "wb") as f: pickle.dump(ensemble_lr_results,  f)


Weight Search: GRU-D + LSTM + LR
Best weights: {'GRU-D': 0.45, 'LSTM': 0.15000000000000002, 'LR': 0.4} | val Sensitivity=0.8823 | val Specificity=0.6757 | val AUPRC=0.4852
Best weights: {'GRU-D': 0.43, 'LSTM': 0.18, 'LR': 0.39} | val Sensitivity=0.8823 | val Specificity=0.6728 | val AUPRC=0.4870
Coarse best: {'GRU-D': 0.45, 'LSTM': 0.15000000000000002, 'LR': 0.4} | val Sensitivity=0.8823 | val Specificity=0.6757
Refined best: {'GRU-D': 0.43, 'LSTM': 0.18, 'LR': 0.39} | val Sensitivity=0.8823 | val Specificity=0.6728 | val AUPRC=0.4870

Ensemble (GRU-D + LSTM + LR)
Fold 1/5 | AUROC=0.8422 | AUPRC=0.4975 | F1=0.4255
Fold 2/5 | AUROC=0.8538 | AUPRC=0.4988 | F1=0.4702
Fold 3/5 | AUROC=0.8700 | AUPRC=0.5221 | F1=0.4211
Fold 4/5 | AUROC=0.8521 | AUPRC=0.5382 | F1=0.4420
Fold 5/5 | AUROC=0.8548 | AUPRC=0.5107 | F1=0.4000

5-Fold CV Summary (mean ± std)
  AUROC         : 0.8546 ± 0.0089
  AUPRC         : 0.5135 ± 0.0153
  F1            : 0.4317 ± 0.0234
  Sensitivity   : 0.8393 ± 0.1073
  Spe

,w_1,w_2,w_3,mean_sensitivity,mean_specificity,mean_auprc,mean_auroc,mean_f1
0,0.45,0.15,0.40,0.882316,0.675711,0.485216,0.840345,0.457115
1,0.50,0.10,0.40,0.879285,0.676676,0.483398,0.839263,0.457411
2,0.25,0.10,0.65,0.879150,0.675209,0.470218,0.838467,0.453351
3,0.50,0.15,0.35,0.876346,0.677175,0.487609,0.839770,0.456988
4,0.25,0.20,0.55,0.876255,0.683439,0.476772,0.841169,0.457353
5,0.10,0.50,0.40,0.876210,0.671834,0.477714,0.840256,0.447500
6,0.00,0.40,0.60,0.876165,0.680531,0.468268,0.838411,0.453961
7,0.35,0.25,0.40,0.873225,0.683923,0.485693,0.841810,0.458138
8,0.10,0.40,0.50,0.873180,0.684401,0.476005,0.840799,0.456694
9,0.05,0.50,0.45,0.873180,0.679084,0.473704,0.839547,0.450812


,w_1,w_2,w_3,mean_sensitivity,mean_specificity,mean_auprc,mean_auroc,mean_f1
0,0.43,0.18,0.39,0.882316,0.672814,0.487017,0.840939,0.455178
1,0.44,0.17,0.39,0.882316,0.673779,0.486751,0.840704,0.455916
2,0.44,0.16,0.40,0.882316,0.675710,0.486741,0.840653,0.457051
3,0.43,0.17,0.40,0.882316,0.673295,0.486519,0.840785,0.455630
4,0.45,0.16,0.39,0.882316,0.674262,0.486381,0.840419,0.456254
5,0.43,0.16,0.41,0.882316,0.675711,0.485913,0.840770,0.456899
6,0.44,0.15,0.41,0.882316,0.675710,0.485473,0.840689,0.457038
7,0.45,0.15,0.40,0.882316,0.675711,0.485216,0.840345,0.457115
8,0.42,0.18,0.40,0.879285,0.676195,0.486610,0.841034,0.455610
9,0.44,0.14,0.42,0.879285,0.676677,0.485005,0.840542,0.456415


,GRU-D,LSTM,LR
0,0.43,0.18,0.39


In [ ]:
ensemble_results = {
    "Ensemble (GRU-D + LSTM + SVM)": ensemble_svm_results,
    "Ensemble (GRU-D + LSTM + LR)": ensemble_lr_results,
}
final_comparison_df = comparison_table(single_model_results, ensemble_results)
display(final_comparison_df)

final_results_payload = {
    "config_source": str(CONFIG_PATH),
    "effective_configs": best_configs,
    "single_models": {
        model_name: {
            "config": result["config"],
            "summary": result["summary"],
        }
        for model_name, result in single_model_results.items()
    },
    "ensemble_weight_search": {
        "Ensemble (GRU-D + LSTM + SVM)": {
            "search_summary": svm_weight_result,
            "top_coarse_candidates": svm_coarse_weight_grid.head(20).to_dict(orient="records"),
            "top_refined_candidates": svm_refined_weight_grid.head(20).to_dict(orient="records"),
        },
        "Ensemble (GRU-D + LSTM + LR)": {
            "search_summary": lr_weight_result,
            "top_coarse_candidates": lr_coarse_weight_grid.head(20).to_dict(orient="records"),
            "top_refined_candidates": lr_refined_weight_grid.head(20).to_dict(orient="records"),
        },
    },
    "ensemble_results": {
        "Ensemble (GRU-D + LSTM + SVM)": {
            "weights": ensemble_svm_results["weights"],
            "summary": ensemble_svm_results["summary"],
        },
        "Ensemble (GRU-D + LSTM + LR)": {
            "weights": ensemble_lr_results["weights"],
            "summary": ensemble_lr_results["summary"],
        },
    },
}

final_results_path = FINAL_OUTPUT_DIR / "final_model_and_ensemble_results.json"
save_json(final_results_path, final_results_payload)
final_comparison_df.to_csv(FINAL_OUTPUT_DIR / "final_comparison_table.csv", index=False)

print(f"Saved final results to: {final_results_path}")
print(f"Saved final comparison table to: {FINAL_OUTPUT_DIR / 'final_comparison_table.csv'}")

,Model,Metric,Mean,Std
0,LSTM,AUROC,0.830342,0.014157
1,LSTM,AUPRC,0.466739,0.022579
2,LSTM,F1,0.425124,0.014377
3,LSTM,Sensitivity,0.821261,0.095488
4,LSTM,Specificity,0.669461,0.071144
5,GRU-D,AUROC,0.828873,0.011140
6,GRU-D,AUPRC,0.472667,0.032558
7,GRU-D,F1,0.451519,0.010717
8,GRU-D,Sensitivity,0.700442,0.081001
9,GRU-D,Specificity,0.773075,0.056133


Saved final results to: /Users/owenmac/Code/0_Project/5925/artifacts/results/dual_ensemble_notebooks/final_training/final_model_and_ensemble_results.json
Saved final comparison table to: /Users/owenmac/Code/0_Project/5925/artifacts/results/dual_ensemble_notebooks/final_training/final_comparison_table.csv
